# VERIONCO — Notebook 04: Evidence Graph & Conflict Detection (Full Dataset, 478 Cases)

**Purpose:** Construct a provenance-aware evidence graph that unifies pathology, genomic, and clinical evidence for the joined cohort produced by Notebook 00 (full dataset: 478 cases / 478 patients), then detect and represent conflicts **without resolving them**. Agents 5 (Evidence Graph) and 6 (Conflict Detection) are implemented here as structured, rule-based, restartable stages.

**Design principles:**
- Uses only artifacts produced by Notebooks 00–03. `patient_id` is the sole join key across modalities.
- No fabricated papers, citations, guidelines, or clinical claims.
- Every node and edge carries provenance (`source_notebook`, `source_file`, `created_at`).
- Conflicts are first-class objects; they are recorded, not auto-resolved.
- Patient-level leakage safety is preserved: all nodes/edges are keyed by `patient_id`.
- Cohort size (`N_CASES`, `N_PATIENTS`) is derived from `cohort_joined_wsi.csv` at run time, never hardcoded, running on the full study cohort of 478 cases.
- Injected-conflict test cases are generated so the detector can be evaluated independently of real data quality.

**Outputs (minimum contract):**
```text
evidence_graph.graphml
evidence_nodes.csv
evidence_edges.csv
conflicts.csv
injected_conflict_tests.json
evidence_graph_qc_report.json
evidence_graph_limitation.json
04_artifacts.zip
```

Downstream notebooks (05–08) consume these artifacts read-only.


## INPUTS

| Source | Path (Kaggle) | Used for |
|--------|---------------|----------|
| Notebook 00 | `/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00/` | `cohort_joined_wsi.csv`, `modality_completeness.csv` |
| Notebook 01 | `/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01/` | `patch_manifest.csv` (schema-tolerant: only columns actually present are used) |
| Notebook 02 | `/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02/` | `patient_mutations.csv`, `gene_level_summary.csv`, `civic_matches_VARIANT_LEVEL.csv`, `civic_matches_GENE_LEVEL_WARNING_NOT_CLINICAL.csv`, `genomic_evidence.json`, `resistance_evidence.csv`, `civic_data_limitation.json` |
| Notebook 03 | `/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03/` | `clinical_structured.json`, `clinical_table.csv`, `clinical_missingness.csv`, `clinical_data_limitation.json` |

**Data-format note:** `cohort_joined_wsi.csv` is one row = one case = one WSI file
(columns: `file_id`, `filename`/`file_name`, `patient_id`, `found`, `n_candidates`,
`roots`, `paths`, `md5_ok`, `size_ok`). There is **no** `case_number` column —
`file_id` is used as the stable per-case identifier instead.

**Leakage contract:** `patient_id` remains the sole join key. No independent splits are introduced.

**Note on dataset slugs:** edit only the `PATHS` block in CONFIGURATION if your
Kaggle attachment slugs differ from the ones above (e.g. after re-publishing
Notebook 03's output as a new Kaggle dataset).


In [1]:
# ============================================================
# CONFIGURATION  ← edit only this cell
# ============================================================
from pathlib import Path
import os, gc, sys, json, hashlib, warnings, time, traceback
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Detect environment
IS_KAGGLE = Path("/kaggle").exists()

def find_dir(candidates, fallback):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return Path(fallback)

OUTPUT_ROOT = Path("/kaggle/working/VERIONCO_outputs") if IS_KAGGLE else Path("./VERIONCO_outputs")
NB_OUTPUT   = OUTPUT_ROOT / "04"
NB_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / "04"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "nb00": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00",
        "/kaggle/input/verionco-outputs-00/00",
        "/kaggle/input/verionco-outputs-00",
        "/kaggle/input/verionco-outputs-local/00",
        "/kaggle/working/VERIONCO_outputs/00",
        "./VERIONCO_outputs/00"
    ], fallback="./VERIONCO_outputs/00" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00"),
    
    "nb01": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01",
        "/kaggle/input/verionco-outputs-01",
        "/kaggle/input/verionco-outputs-local/01",
        "/kaggle/working/VERIONCO_outputs/01",
        "./VERIONCO_outputs/01"
    ], fallback="./VERIONCO_outputs/01" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01"),
    
    "nb02": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02",
        "/kaggle/input/verionco-outputs-02",
        "/kaggle/input/verionco-outputs-local/02",
        "/kaggle/working/VERIONCO_outputs/02",
        "./VERIONCO_outputs/02"
    ], fallback="./VERIONCO_outputs/02" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02"),
    
    "nb03": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03",
        "/kaggle/input/verionco-outputs-03",
        "/kaggle/input/verionco-outputs-local/03",
        "/kaggle/working/VERIONCO_outputs/03",
        "./VERIONCO_outputs/03"
    ], fallback="./VERIONCO_outputs/03" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03"),
}

EXPECTED_MIN_CASES = 10
EXPECTED_MAX_CASES = 550
PROJECT  = "VERIONCO"
NOTEBOOK = "04_evidence_graph_conflict"

for k, v in PATHS.items():
    print(f"{k:8s} | {v} | exists={v.exists()}")

critical = ["nb00", "nb02", "nb03"]
missing_critical = [k for k in critical if not PATHS[k].exists()]
assert not missing_critical, f"Critical upstream missing: {missing_critical}"

if not PATHS["nb01"].exists():
    warnings.warn(f"Notebook 01 root missing: {PATHS['nb01']} — running in pathology-degraded mode.")

print(f"\nOutput:      {NB_OUTPUT}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")


nb00     | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00 | exists=False
nb01     | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01 | exists=False
nb02     | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02 | exists=False
nb03     | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03 | exists=False

Output:      /kaggle/working/VERIONCO_outputs/04
Checkpoints: /kaggle/working/VERIONCO_outputs/checkpoints/04
Caps: mut=80/patient, resistance=30/patient
CIViC gene-level fraction threshold: 0.95


/tmp/ipykernel_16/1290588945.py:52: UserWarning: Critical roots missing: ['nb00', 'nb02', 'nb03']. Attach the corresponding Kaggle datasets before running.
  warnings.warn(f"Critical roots missing: {missing_critical}. "
/tmp/ipykernel_16/1290588945.py:55: UserWarning: Notebook 01 root missing: /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01 — pathology_summary nodes will be skipped.
  warnings.warn(f"Notebook 01 root missing: {PATHS['nb01']} — "


In [2]:
# ============================================================
# ENVIRONMENT
# ============================================================
import importlib.util
packages = ["pandas", "numpy", "matplotlib", "seaborn", "networkx"]
rows = [{"package": p, "available": importlib.util.find_spec(p) is not None} for p in packages]
display(pd.DataFrame(rows))

runtime = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
}
try:
    import networkx as nx
    runtime["networkx_version"] = nx.__version__
except Exception as e:
    runtime["networkx_error"] = str(e)

print(json.dumps(runtime, indent=2))

,package,available
0,pandas,True
1,numpy,True
2,matplotlib,True
3,seaborn,True
4,networkx,True


{
  "timestamp_utc": "2026-09-14T20:16:09.978912+00:00",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "networkx_version": "3.6.1"
}


## DATA VALIDATION

1. Load Notebook 00–03 artifacts using the real filenames each notebook actually
   produced (`cohort_joined_wsi.csv`, not the retired `case_manifest_validated.csv`).
2. Derive `N_CASES` / `N_PATIENTS` from the loaded cohort — no hardcoded cohort size.
3. Confirm `patient_id` consistency across modalities (mutations, clinical, cohort).
4. Surface documented limitations from prior notebooks (CIViC granularity, clinical missingness).


In [3]:
# ============================================================
# DATA VALIDATION — load upstream artifacts (real filenames/schemas)
# ============================================================
t0 = time.time()

def _safe_read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"  MISSING: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)

def _safe_read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"  MISSING: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# --- Notebook 00: cohort (1 row = 1 case = 1 WSI) ------------
cohort_path = PATHS["nb00"] / "cohort_joined_wsi.csv"
assert cohort_path.exists(), f"Missing {cohort_path}. Run Notebook 00 first."
manifest = pd.read_csv(cohort_path)
if "filename" in manifest.columns and "file_name" not in manifest.columns:
    manifest = manifest.rename(columns={"filename": "file_name"})
assert "patient_id" in manifest.columns, "cohort_joined_wsi.csv missing patient_id"
assert manifest["patient_id"].isna().sum() == 0, "cohort_joined_wsi.csv has missing patient_id"

patient_set = set(manifest["patient_id"].astype(str).unique())
N_CASES     = len(manifest)
N_PATIENTS  = len(patient_set)
print(f"[00] cohort_joined_wsi.csv: {manifest.shape} | "
      f"{N_CASES} cases | {N_PATIENTS} unique patients")
if N_CASES != N_PATIENTS:
    print(f"  NOTE: {N_CASES} cases map to {N_PATIENTS} patients "
          f"(some patients have >1 case) — this is expected, not an error.")

mod_comp = _safe_read_csv(PATHS["nb00"] / "modality_completeness.csv")
if len(mod_comp):
    print(f"[00] modality_completeness.csv: {mod_comp.shape}")

# --- Notebook 01: pathology (schema-tolerant) -----------------
patch_summary = pd.DataFrame()
if PATHS["nb01"].exists():
    hits = list(PATHS["nb01"].rglob("patch_manifest.csv"))
    if hits:
        patch_raw = pd.read_csv(hits[0])
        print(f"[01] patch_manifest.csv: {patch_raw.shape} | columns: {list(patch_raw.columns)}")
        if "patient_id" in patch_raw.columns:
            # Schema-tolerant aggregation: count rows per patient as n_patches,
            # and average any of the known optional numeric QC columns if present.
            agg_spec = {"file_name": ("file_name", "first")} if "file_name" in patch_raw.columns else {}
            for opt_col, agg_name in [("tissue_fraction", "mean_tissue"),
                                       ("mean_tissue", "mean_tissue"),
                                       ("mean_intensity", "mean_intensity")]:
                if opt_col in patch_raw.columns and agg_name not in agg_spec:
                    agg_spec[agg_name] = (opt_col, "mean")
            grouped = patch_raw.groupby("patient_id").agg(
                n_patches=("patient_id", "size"), **agg_spec
            ).reset_index()
            if "case_number" not in grouped.columns:
                grouped["case_number"] = None
            patch_summary = grouped
    else:
        print("[01] No patch_manifest.csv found under NB01 root.")
else:
    print("[01] NB01 root not attached — pathology_summary nodes will be skipped.")

# --- Notebook 02: genomics -------------------------------------
mutations    = _safe_read_csv(PATHS["nb02"] / "patient_mutations.csv", low_memory=False)
gene_summary = _safe_read_csv(PATHS["nb02"] / "gene_level_summary.csv")

civic_variant = _safe_read_csv(PATHS["nb02"] / "civic_matches_VARIANT_LEVEL.csv")
civic_gene    = _safe_read_csv(PATHS["nb02"] / "civic_matches_GENE_LEVEL_WARNING_NOT_CLINICAL.csv")
civic_matches = pd.concat([civic_variant, civic_gene], ignore_index=True) \
                if len(civic_variant) or len(civic_gene) else pd.DataFrame()

genomic_ev = _safe_read_json(PATHS["nb02"] / "genomic_evidence.json")
resistance = _safe_read_csv(PATHS["nb02"] / "resistance_evidence.csv", low_memory=False)
civic_lim  = _safe_read_json(PATHS["nb02"] / "civic_data_limitation.json")

print(f"[02] mutations: {mutations.shape} | genes: {gene_summary.shape} | "
      f"CIViC matches: {civic_matches.shape} "
      f"(variant={len(civic_variant)}, gene={len(civic_gene)}) | "
      f"resistance: {resistance.shape}")

# --- Notebook 03: clinical --------------------------------------
clinical_struct = _safe_read_json(PATHS["nb03"] / "clinical_structured.json")
clinical_table  = _safe_read_csv(PATHS["nb03"] / "clinical_table.csv")
clin_missing    = _safe_read_csv(PATHS["nb03"] / "clinical_missingness.csv")
clin_lim        = _safe_read_json(PATHS["nb03"] / "clinical_data_limitation.json")
print(f"[03] clinical_table: {clinical_table.shape} | structured patients: {len(clinical_struct)}")

# --- Cross-modality patient consistency --------------------------
mut_patients  = set(mutations["patient_id"].dropna().astype(str).unique()) if len(mutations) else set()
clin_patients = set(clinical_table["patient_id"].dropna().astype(str).unique()) if len(clinical_table) else set()
print(f"\nPatient set intersection (cohort ∩ mut ∩ clin): "
      f"{len(patient_set & mut_patients & clin_patients)} / {N_PATIENTS}")

if mut_patients and mut_patients != patient_set:
    print(f"  NOTE: mutation-patient set differs from cohort by "
          f"{len(mut_patients ^ patient_set)} patient(s).")
if clin_patients and clin_patients != patient_set:
    missing_clin = sorted(patient_set - clin_patients)
    print(f"  NOTE: {len(missing_clin)} cohort patients have no clinical record "
          f"(consistent with NB03's documented clinical_data_limitation.json).")

print(f"\nValidation elapsed: {time.time()-t0:.1f}s")
display(manifest.head(10))


AssertionError: Missing /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00/cohort_joined_wsi.csv. Run Notebook 00 first.

## PROCESSING — Evidence Graph Construction

Graph model:
- **Directed** `MultiDiGraph` (NetworkX) so multiple edge types between the same pair are allowed.
- Node types: `patient`, `case`, `wsi`, `mutation`, `gene`, `gene_summary`, `clinical_finding`, `clinical_summary`, `civic_evidence`, `resistance_signal`, `pathology_summary`, `conflict`.
- Edge types: `has_case`, `observed_in`, `has_mutation`, `affects_gene`, `has_gene_summary`, `aggregates`, `has_clinical`, `has_clinical_summary`, `has_pathology_summary`, `matched_to`, `evidence_for`, `suggests_resistance`, `derived_from`, `involved_in_conflict`, `has_conflict`.
- Every node/edge stores provenance fields (`source_notebook`, `source_file`, `created_at`).
- Case/WSI identity comes from `file_id` in `cohort_joined_wsi.csv` (there is no `case_number` column upstream).

No recommendation or guideline nodes are created here (those appear in Notebooks 06–07).
Caps: mutation nodes capped at `MAX_MUT_NODES_PER_PATIENT` per patient; resistance nodes capped at `RESISTANCE_NODE_CAP_PER_PATIENT` per patient. Both caps are recorded in the QC report.


In [ ]:
# ============================================================
# PROCESSING — build evidence graph (nodes + edges)
# ============================================================
import networkx as nx

G = nx.MultiDiGraph(name="VERIONCO_evidence_graph", cohort="TCGA-LUAD", n_cases=N_CASES)

node_records = []
edge_records = []

def add_node(nid, ntype, **attrs):
    attrs = {k: v for k, v in attrs.items()
             if v is not None and (not isinstance(v, float) or not np.isnan(v))}
    attrs["node_type"] = ntype
    attrs.setdefault("provenance_source", NOTEBOOK)
    attrs.setdefault("created_at", datetime.now(timezone.utc).isoformat())
    G.add_node(nid, **attrs)
    rec = {"node_id": nid, "node_type": ntype, **{k: str(v)[:500] for k, v in attrs.items()}}
    node_records.append(rec)

def add_edge(src, dst, etype, **attrs):
    attrs = {k: v for k, v in attrs.items()
             if v is not None and (not isinstance(v, float) or not np.isnan(v))}
    attrs["edge_type"] = etype
    attrs.setdefault("provenance_source", NOTEBOOK)
    G.add_edge(src, dst, key=etype, **attrs)
    rec = {"source": src, "target": dst, "edge_type": etype,
           **{k: str(v)[:500] for k, v in attrs.items()}}
    edge_records.append(rec)

ts = datetime.now(timezone.utc).isoformat()

# ---------- Patients ----------
for pid in sorted(patient_set):
    add_node(
        f"patient:{pid}", "patient",
        patient_id=pid,
        label=pid,
        source_notebook="00",
        source_file="cohort_joined_wsi.csv",
    )

# ---------- Cases / WSI ----------
# cohort_joined_wsi.csv is 1 row = 1 case = 1 WSI file; file_id is the stable
# per-case identifier (there is no case_number column upstream).
for _, row in manifest.iterrows():
    pid     = row["patient_id"]
    file_id = row.get("file_id", row.get("file_name"))
    case_id = f"case:{file_id}"
    wsi_id  = f"wsi:{row.get('file_name', file_id)}"
    add_node(
        case_id, "case",
        patient_id=pid,
        file_id=file_id,
        file_name=row.get("file_name"),
        found=row.get("found"),
        n_candidates=row.get("n_candidates"),
        label=f"Case {file_id}",
        source_notebook="00",
        source_file="cohort_joined_wsi.csv",
    )
    add_node(
        wsi_id, "wsi",
        patient_id=pid,
        file_id=file_id,
        file_name=row.get("file_name"),
        label=str(row.get("file_name", file_id))[:60],
        source_notebook="00",
        source_file="cohort_joined_wsi.csv",
    )
    add_edge(f"patient:{pid}", case_id, "has_case",
             patient_id=pid, source_notebook="00")
    add_edge(case_id, wsi_id, "observed_in",
             patient_id=pid, source_notebook="00")

# ---------- Pathology summaries (from NB01, schema-tolerant) ----------
if len(patch_summary):
    optional_cols = [c for c in ("mean_tissue", "mean_intensity", "file_name")
                      if c in patch_summary.columns]
    for _, row in patch_summary.iterrows():
        pid = row.get("patient_id")
        if pid not in patient_set:
            continue
        case_no = row.get("case_number")
        ps_id = f"pathology:{pid}:{case_no if pd.notna(case_no) else 0}"
        attrs = {c: row.get(c) for c in optional_cols}
        add_node(
            ps_id, "pathology_summary",
            patient_id=pid,
            n_patches=row.get("n_patches"),
            label=f"Pathology summary {pid}",
            source_notebook="01",
            source_file="patch_manifest.csv",
            **attrs,
        )
        add_edge(f"patient:{pid}", ps_id, "has_pathology_summary",
                 patient_id=pid, source_notebook="01")

print(f"Patients + cases + WSI + pathology nodes: {G.number_of_nodes()}")


In [ ]:
# ---------- Genomic nodes (mutations, genes, CIViC, resistance) ----------
def _parse_genes_mentioned(raw):
    """Safely parse genes_mentioned from resistance_evidence.csv (no eval)."""
    if isinstance(raw, list):
        genes_list = raw
    elif isinstance(raw, str):
        raw_s = raw.strip()
        try:
            if raw_s.startswith("["):
                genes_list = ast.literal_eval(raw_s)
            else:
                genes_list = [g.strip() for g in raw_s.split(",") if g.strip()]
        except (ValueError, SyntaxError):
            genes_list = [raw_s]
    else:
        return []
    if not isinstance(genes_list, list):
        genes_list = [genes_list]
    return [str(g).upper() for g in genes_list if g is not None]


if len(mutations):
    for pid in sorted(patient_set):
        p_muts = mutations[mutations["patient_id"] == pid]
        if "tumor_vaf" in p_muts.columns:
            p_muts = p_muts.sort_values("tumor_vaf", ascending=False, na_position="last")
        p_muts = p_muts.head(MAX_MUT_NODES_PER_PATIENT)

        for idx, row in p_muts.iterrows():
            gene = str(row.get("Hugo_Symbol", "UNK")).strip()
            var  = str(row.get("variant_canonical", row.get("HGVSp_Short", ""))).strip()
            mut_id = f"mutation:{pid}:{gene}:{hashlib.md5(var.encode()).hexdigest()[:10]}"
            add_node(
                mut_id, "mutation",
                patient_id=pid,
                gene=gene,
                variant=var[:120],
                variant_classification=row.get("Variant_Classification"),
                tumor_vaf=row.get("tumor_vaf"),
                label=f"{gene} {var[:40]}",
                source_notebook="02",
                source_file="patient_mutations.csv",
            )
            add_edge(f"patient:{pid}", mut_id, "has_mutation",
                     patient_id=pid, source_notebook="02")
            gene_id = f"gene:{gene}"
            if gene_id not in G:
                add_node(gene_id, "gene", gene=gene, label=gene,
                         source_notebook="02", source_file="patient_mutations.csv")
            add_edge(mut_id, gene_id, "affects_gene",
                     patient_id=pid, source_notebook="02")

# Gene-level summary nodes
if len(gene_summary):
    for _, row in gene_summary.iterrows():
        pid = row["patient_id"]
        if pid not in patient_set:
            continue
        gene = str(row["Hugo_Symbol"]).strip()
        gs_id = f"gene_summary:{pid}:{gene}"
        add_node(
            gs_id, "gene_summary",
            patient_id=pid,
            gene=gene,
            n_variants=row.get("n_variants"),
            label=f"{pid} {gene} (n={row.get('n_variants')})",
            source_notebook="02",
            source_file="gene_level_summary.csv",
        )
        add_edge(f"patient:{pid}", gs_id, "has_gene_summary",
                 patient_id=pid, source_notebook="02")
        gene_id = f"gene:{gene}"
        if gene_id not in G:
            add_node(gene_id, "gene", gene=gene, label=gene,
                     source_notebook="02")
        add_edge(gs_id, gene_id, "aggregates",
                 patient_id=pid, source_notebook="02")

# CIViC matches
if len(civic_matches):
    for _, row in civic_matches.iterrows():
        pid = row.get("patient_id")
        if pid not in patient_set:
            continue
        gene = str(row.get("gene", row.get("Hugo_Symbol", "UNK"))).strip()
        match_level = str(row.get("match_level", "gene")).lower()
        civic_id = f"civic:{pid}:{gene}:{hashlib.md5(str(row.to_dict()).encode()).hexdigest()[:8]}"
        add_node(
            civic_id, "civic_evidence",
            patient_id=pid,
            gene=gene,
            match_level=match_level,
            label=f"CIViC {gene} ({match_level})",
            source_notebook="02",
            source_file="civic_matches_VARIANT_LEVEL.csv / civic_matches_GENE_LEVEL_WARNING_NOT_CLINICAL.csv",
            limitation="gene-level where variant-level unavailable (see civic_data_limitation.json)",
        )
        add_edge(f"patient:{pid}", civic_id, "matched_to",
                 patient_id=pid, match_level=match_level, source_notebook="02")
        gene_id = f"gene:{gene}"
        if gene_id in G:
            add_edge(civic_id, gene_id, "evidence_for",
                     patient_id=pid, match_level=match_level, source_notebook="02")

# Resistance signals (associate to patients carrying mentioned genes; capped)
if len(resistance) and "genes_mentioned" in resistance.columns:
    for pid in sorted(patient_set):
        p_genes = set()
        if len(gene_summary):
            p_genes = set(
                gene_summary[gene_summary["patient_id"] == pid]["Hugo_Symbol"]
                .astype(str).str.upper()
            )
        if not p_genes:
            continue
        matched = 0
        for _, row in resistance.iterrows():
            if matched >= RESISTANCE_NODE_CAP_PER_PATIENT:
                break
            genes_list = _parse_genes_mentioned(row.get("genes_mentioned", []))
            if not genes_list:
                continue
            if p_genes & set(genes_list):
                res_id = f"resistance:{pid}:{hashlib.md5(str(row.to_dict()).encode()).hexdigest()[:8]}"
                add_node(
                    res_id, "resistance_signal",
                    patient_id=pid,
                    genes=genes_list[:10],
                    source_type=row.get("source_type", row.get("source", "unknown")),
                    label=f"Resistance signal {pid}",
                    source_notebook="02",
                    source_file="resistance_evidence.csv",
                )
                add_edge(f"patient:{pid}", res_id, "suggests_resistance",
                         patient_id=pid, source_notebook="02")
                matched += 1

print(f"After genomic layer: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")


In [ ]:
# ---------- Clinical findings ----------
CLINICAL_FIELDS_OF_INTEREST = [
    "demographics__AGE", "demographics__SEX",
    "staging__AJCC_PATHOLOGIC_TUMOR_STAGE", "staging__PATH_T_STAGE",
    "staging__PATH_N_STAGE", "staging__PATH_M_STAGE",
    "diagnosis__SUBTYPE", "diagnosis__CANCER_TYPE_ACRONYM",
    "diagnosis__ICD_10",
    "sample_diagnosis__CANCER_TYPE", "sample_diagnosis__CANCER_TYPE_DETAILED",
    "sample_diagnosis__ONCOTREE_CODE", "sample_diagnosis__GRADE",
    "survival__OS_STATUS", "survival__OS_MONTHS",
    "survival__PFS_STATUS", "survival__PFS_MONTHS",
]

if len(clinical_table):
    for _, row in clinical_table.iterrows():
        pid = row.get("patient_id")
        if pid not in patient_set:
            continue
        for col in CLINICAL_FIELDS_OF_INTEREST:
            if col not in clinical_table.columns:
                continue
            val = row[col]
            if pd.isna(val) or str(val).strip() in {"", "[Not Available]", "nan", "None"}:
                continue
            cf_id = f"clinical:{pid}:{col}"
            add_node(
                cf_id, "clinical_finding",
                patient_id=pid,
                field=col,
                value=str(val)[:120],
                label=f"{col.split('__')[-1]}={val}",
                source_notebook="03",
                source_file="clinical_table.csv",
            )
            add_edge(f"patient:{pid}", cf_id, "has_clinical",
                     patient_id=pid, field=col, source_notebook="03")

# High-level clinical summary per patient
for pid in sorted(patient_set):
    if pid not in clinical_struct:
        continue
    cs = clinical_struct[pid]
    cs_id = f"clinical_summary:{pid}"
    stage = None
    if isinstance(cs.get("staging"), dict):
        stage = cs["staging"].get("AJCC_PATHOLOGIC_TUMOR_STAGE") or cs["staging"].get("PATH_T_STAGE")
    age = None
    sex = None
    if isinstance(cs.get("demographics"), dict):
        age = cs["demographics"].get("AGE")
        sex = cs["demographics"].get("SEX")
    add_node(
        cs_id, "clinical_summary",
        patient_id=pid,
        stage=stage,
        age=age,
        sex=sex,
        label=f"Clinical summary {pid}",
        source_notebook="03",
        source_file="clinical_structured.json",
    )
    add_edge(f"patient:{pid}", cs_id, "has_clinical_summary",
             patient_id=pid, source_notebook="03")

print(f"Final pre-conflict graph: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")
type_counts = pd.Series([d.get("node_type") for _, d in G.nodes(data=True)]).value_counts()
display(type_counts.to_frame("count"))

## PROCESSING — Conflict Detection

Conflicts are **represented**, never auto-resolved. Each conflict record contains:
- `conflict_id`, `patient_id`, `conflict_type`
- `entity_a`, `entity_b` (node ids)
- `description`, `severity` (low / medium / high)
- `evidence_a`, `evidence_b`
- `provenance`

Conflict types implemented (explicit):
1. `high_mutation_burden` — extreme somatic burden (possible technical or biological flag)
2. `civic_gene_only` — CIViC match exists only at gene level (not variant-specific)
3. `missing_clinical_stage` — staging field absent
4. `missing_key_demographics` — age/sex missing
5. `resistance_signal_present` — resistance literature/trial signal linked to patient genes (reports **true count** + capped node count)
6. `pathology_genomic_coverage` — patient has pathology but very few filtered mutations (or vice versa)
7. `injected_*` — synthetic test cases (see next section)

These are **not** clinical diagnoses; they are machine-checkable consistency flags for the safety gate and planners.

In [ ]:
# ============================================================
# PROCESSING — conflict detection (rule-based)  [BUGFIX]
# Fixes:
#   1. civic_gene_only uses gene-level fraction threshold (not 100% pure)
#   2. genes_mentioned parsed with ast.literal_eval (no bare eval)
#   3. detect_patient_conflicts shared with injected tests
# ============================================================

conflicts = []

def record_conflict(patient_id, conflict_type, entity_a, entity_b,
                    description, severity="medium", evidence_a=None, evidence_b=None,
                    extra=None, is_injected_test=False):
    cid = f"conflict:{patient_id}:{conflict_type}:{len(conflicts):04d}"
    rec = {
        "conflict_id": cid,
        "patient_id": patient_id,
        "conflict_type": conflict_type,
        "entity_a": entity_a,
        "entity_b": entity_b,
        "description": description,
        "severity": severity,
        "evidence_a": evidence_a,
        "evidence_b": evidence_b,
        "source_notebook": NOTEBOOK,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "is_injected_test": bool(is_injected_test),
    }
    if extra:
        rec.update(extra)
    conflicts.append(rec)
    add_node(
        cid, "conflict",
        patient_id=patient_id,
        conflict_type=conflict_type,
        severity=severity,
        is_injected_test=bool(is_injected_test),
        label=f"{conflict_type} ({severity})",
        description=description[:200],
        source_notebook=NOTEBOOK,
    )
    if entity_a and entity_a in G:
        add_edge(entity_a, cid, "involved_in_conflict", patient_id=patient_id)
    if entity_b and entity_b in G:
        add_edge(entity_b, cid, "involved_in_conflict", patient_id=patient_id)
    if f"patient:{patient_id}" in G:
        add_edge(f"patient:{patient_id}", cid, "has_conflict", patient_id=patient_id)


def detect_patient_conflicts(pid, G, mutations, civic_matches, clinical_table,
                             gene_summary, resistance,
                             high_mut_burden=None,
                             civic_gene_frac=None,
                             resistance_cap=None):
    """Shared detector for real cohort and injected unit tests."""
    if high_mut_burden is None:
        high_mut_burden = HIGH_MUTATION_BURDEN
    if civic_gene_frac is None:
        civic_gene_frac = CIVIC_GENE_LEVEL_FRACTION
    if resistance_cap is None:
        resistance_cap = RESISTANCE_NODE_CAP_PER_PATIENT

    found = []

    # 1. Mutation burden
    n_mut = int((mutations["patient_id"] == pid).sum()) if len(mutations) else 0
    if n_mut >= high_mut_burden:
        found.append({
            "conflict_type": "high_mutation_burden",
            "entity_a": f"patient:{pid}",
            "entity_b": None,
            "description": (
                f"Patient has {n_mut} filtered mutations (threshold={high_mut_burden}). "
                "May indicate hypermutation, technical artifact, or need for review."
            ),
            "severity": "high" if n_mut >= 1000 else "medium",
            "evidence_a": f"n_mutations={n_mut}",
            "evidence_b": None,
        })

    # 2. CIViC gene-level dominance (BUGFIX: fraction threshold, not 100%)
    if len(civic_matches) and "patient_id" in civic_matches.columns:
        p_civic = civic_matches[civic_matches["patient_id"] == pid]
        if len(p_civic):
            if "match_level" in p_civic.columns:
                levels = p_civic["match_level"].astype(str).str.lower().str.strip()
            else:
                levels = pd.Series(["gene"] * len(p_civic))
            n_gene = int((levels == "gene").sum())
            n_var  = int((levels == "variant").sum())
            n_tot  = len(p_civic)
            frac_gene = (n_gene / n_tot) if n_tot else 0.0
            if n_tot > 0 and (n_var == 0 or frac_gene >= civic_gene_frac):
                found.append({
                    "conflict_type": "civic_gene_only",
                    "entity_a": f"patient:{pid}",
                    "entity_b": None,
                    "description": (
                        f"CIViC matches for this patient are predominantly gene-level "
                        f"({n_gene}/{n_tot} = {frac_gene:.1%}; variant-level={n_var}). "
                        "Do not treat gene-level matches as variant-specific clinical evidence "
                        "(see civic_data_limitation.json)."
                    ),
                    "severity": "medium",
                    "evidence_a": f"n_gene={n_gene}, n_variant={n_var}, frac_gene={frac_gene:.3f}",
                    "evidence_b": None,
                    "extra": {
                        "n_civic_gene_level": n_gene,
                        "n_civic_variant_level": n_var,
                        "civic_gene_fraction": frac_gene,
                    },
                })

    # 3. Missing clinical stage
    stage_val = None
    if len(clinical_table) and "staging__AJCC_PATHOLOGIC_TUMOR_STAGE" in clinical_table.columns:
        row = clinical_table[clinical_table["patient_id"] == pid]
        if len(row):
            stage_val = row.iloc[0].get("staging__AJCC_PATHOLOGIC_TUMOR_STAGE")
    stage_missing = (
        stage_val is None
        or (isinstance(stage_val, float) and np.isnan(stage_val))
        or str(stage_val).strip() in {"", "[Not Available]", "nan", "None"}
    )
    if stage_missing:
        found.append({
            "conflict_type": "missing_clinical_stage",
            "entity_a": f"patient:{pid}",
            "entity_b": f"clinical_summary:{pid}" if f"clinical_summary:{pid}" in G else None,
            "description": (
                "AJCC pathologic tumor stage is missing or unavailable. "
                "Guideline and trial planners must not assume stage."
            ),
            "severity": "high",
            "evidence_a": str(stage_val),
            "evidence_b": None,
        })

    # 4. Missing key demographics
    age_val = sex_val = None
    if len(clinical_table):
        row = clinical_table[clinical_table["patient_id"] == pid]
        if len(row):
            if "demographics__AGE" in clinical_table.columns:
                age_val = row.iloc[0].get("demographics__AGE")
            if "demographics__SEX" in clinical_table.columns:
                sex_val = row.iloc[0].get("demographics__SEX")
    missing_demo = []
    if age_val is None or (isinstance(age_val, float) and np.isnan(age_val)):
        missing_demo.append("AGE")
    if sex_val is None or str(sex_val).strip() in {"", "nan", "[Not Available]", "None"}:
        missing_demo.append("SEX")
    if missing_demo:
        found.append({
            "conflict_type": "missing_key_demographics",
            "entity_a": f"patient:{pid}",
            "entity_b": None,
            "description": (
                f"Missing demographics: {missing_demo}. "
                "Patient-specific planner must not invent values."
            ),
            "severity": "medium",
            "evidence_a": str(missing_demo),
            "evidence_b": None,
        })

    # 5. Resistance signals present
    res_nodes = [
        n for n, d in G.nodes(data=True)
        if d.get("node_type") == "resistance_signal" and d.get("patient_id") == pid
    ]
    p_genes = set()
    if len(gene_summary):
        p_genes = set(
            gene_summary[gene_summary["patient_id"] == pid]["Hugo_Symbol"]
            .astype(str).str.upper()
        )
    true_res_count = 0
    if p_genes and len(resistance) and "genes_mentioned" in resistance.columns:
        for _, row in resistance.iterrows():
            glist = _parse_genes_mentioned(row.get("genes_mentioned", []))
            if p_genes & set(glist):
                true_res_count += 1
    if true_res_count > 0 or res_nodes:
        found.append({
            "conflict_type": "resistance_signal_present",
            "entity_a": f"patient:{pid}",
            "entity_b": res_nodes[0] if res_nodes else None,
            "description": (
                f"{true_res_count} resistance records match this patient's mutated genes; "
                f"{len(res_nodes)} materialised as graph nodes "
                f"(capped at {resistance_cap} per patient). "
                "Counter-evidence retrieval (NB05) and red-team (NB07) must consider them."
            ),
            "severity": "medium",
            "evidence_a": f"true_count={true_res_count}",
            "evidence_b": f"graph_node_count={len(res_nodes)}",
            "extra": {"true_resistance_count": true_res_count},
        })

    # 6. Pathology vs genomic coverage
    has_path = any(
        d.get("node_type") == "pathology_summary" and d.get("patient_id") == pid
        for _, d in G.nodes(data=True)
    )
    if has_path and n_mut == 0:
        found.append({
            "conflict_type": "pathology_genomic_coverage",
            "entity_a": f"patient:{pid}",
            "entity_b": None,
            "description": (
                "Pathology evidence present but zero filtered mutations retained. "
                "Possible filtering artifact or true low-mutation case; do not invent variants."
            ),
            "severity": "medium",
            "evidence_a": f"n_mutations={n_mut}",
            "evidence_b": "pathology_present=True",
        })
    elif (not has_path) and n_mut > 0:
        found.append({
            "conflict_type": "pathology_genomic_coverage",
            "entity_a": f"patient:{pid}",
            "entity_b": None,
            "description": (
                "Genomic evidence present but no pathology summary node. Check NB01 handoff."
            ),
            "severity": "low",
            "evidence_a": f"n_mutations={n_mut}",
            "evidence_b": "pathology_present=False",
        })

    return found


# --- Run on real cohort ---
for pid in sorted(patient_set):
    for c in detect_patient_conflicts(
        pid, G, mutations, civic_matches, clinical_table,
        gene_summary, resistance
    ):
        record_conflict(
            patient_id=pid,
            conflict_type=c["conflict_type"],
            entity_a=c.get("entity_a"),
            entity_b=c.get("entity_b"),
            description=c["description"],
            severity=c.get("severity", "medium"),
            evidence_a=c.get("evidence_a"),
            evidence_b=c.get("evidence_b"),
            extra=c.get("extra"),
            is_injected_test=False,
        )

conflicts_df = pd.DataFrame(conflicts)
print(f"Detected conflicts (real): {len(conflicts_df)}")
if len(conflicts_df):
    display(conflicts_df.groupby(["conflict_type", "severity"]).size().reset_index(name="count"))
    display(conflicts_df[["conflict_id", "patient_id", "conflict_type", "severity", "description"]].head(30))


## Injected-conflict test cases

Synthetic conflicts are injected **on a copy of the graph** (or as separate records) so that the detector can be unit-tested without polluting real patient evidence. Each test case states the expected conflict type and whether the detector should fire.

In [ ]:
# ============================================================
# INJECTED CONFLICT TESTS  [BUGFIX — uses shared detect_patient_conflicts]
# ============================================================
injected_tests = [
    {
        "test_id": "inj_001_high_burden",
        "description": "Synthetic patient with 1200 mutations should trigger high_mutation_burden",
        "setup": {"patient_id": "SYNTH-HIGH", "n_mutations": 1200},
        "expected_conflict_type": "high_mutation_burden",
        "expected_severity": "high",
        "should_detect": True,
    },
    {
        "test_id": "inj_002_civic_gene_only",
        "description": "Patient whose CIViC matches are gene-level dominated should trigger civic_gene_only",
        "setup": {
            "patient_id": "SYNTH-CIVIC",
            "civic_rows": [{"match_level": "gene"}] * 20 + [{"match_level": "variant"}] * 1,
        },
        "expected_conflict_type": "civic_gene_only",
        "expected_severity": "medium",
        "should_detect": True,
    },
    {
        "test_id": "inj_003_missing_stage",
        "description": "Patient with no AJCC stage should trigger missing_clinical_stage",
        "setup": {"patient_id": "SYNTH-STAGE", "stage": None},
        "expected_conflict_type": "missing_clinical_stage",
        "expected_severity": "high",
        "should_detect": True,
    },
    {
        "test_id": "inj_004_negative_burden",
        "description": "Patient with 50 mutations should NOT trigger high_mutation_burden",
        "setup": {"patient_id": "SYNTH-LOW", "n_mutations": 50},
        "expected_conflict_type": "high_mutation_burden",
        "should_detect": False,
    },
]


def run_injected_setup(setup):
    """Build minimal tables and call the real shared detector."""
    pid = setup["patient_id"]
    mut = pd.DataFrame(columns=["patient_id"])
    if "n_mutations" in setup:
        mut = pd.DataFrame({"patient_id": [pid] * int(setup["n_mutations"])})

    civic = pd.DataFrame(columns=["patient_id", "match_level"])
    if "civic_rows" in setup:
        civic = pd.DataFrame(setup["civic_rows"])
        civic["patient_id"] = pid

    clin = pd.DataFrame({"patient_id": [pid]})
    if "stage" in setup:
        clin["staging__AJCC_PATHOLOGIC_TUMOR_STAGE"] = setup["stage"]
    else:
        clin["staging__AJCC_PATHOLOGIC_TUMOR_STAGE"] = "Stage IIA"
    clin["demographics__AGE"] = 60
    clin["demographics__SEX"] = "Male"

    genes = pd.DataFrame(columns=["patient_id", "Hugo_Symbol"])
    res = pd.DataFrame(columns=["genes_mentioned"])
    G_tmp = nx.MultiDiGraph()
    G_tmp.add_node(f"patient:{pid}", node_type="patient", patient_id=pid)

    return detect_patient_conflicts(pid, G_tmp, mut, civic, clin, genes, res)


test_results = []
for t in injected_tests:
    fired = run_injected_setup(t["setup"])
    fired_types = [c["conflict_type"] for c in fired]
    expected = t["expected_conflict_type"]
    detected = expected in fired_types
    passed = detected == t["should_detect"]
    test_results.append({
        **t,
        "fired_types": fired_types,
        "detected": detected,
        "passed": passed,
    })
    if t["should_detect"] and detected:
        hit = next(c for c in fired if c["conflict_type"] == expected)
        record_conflict(
            patient_id=t["setup"]["patient_id"],
            conflict_type=f"injected_{expected}",
            entity_a=None,
            entity_b=None,
            description=t["description"],
            severity=t.get("expected_severity", hit.get("severity", "medium")),
            evidence_a=str(t["setup"]),
            is_injected_test=True,
        )

test_results_df = pd.DataFrame(test_results)
print("Injected test results:")
display(test_results_df[["test_id", "expected_conflict_type", "should_detect", "detected", "passed"]])
print(f"Passed: {int(test_results_df['passed'].sum())} / {len(test_results_df)}")

with open(NB_OUTPUT / "injected_conflict_tests.json", "w", encoding="utf-8") as f:
    json.dump({"tests": injected_tests, "results": test_results}, f, indent=2, default=str)


## OUTPUTS — persist graph, tables, QC

In [ ]:
# ============================================================
# OUTPUTS — graph + tables (recompute type_counts AFTER all mutations)
# ============================================================
# Refresh dataframes from final graph state
node_records = []
for nid, data in G.nodes(data=True):
    rec = {"node_id": nid, **{k: (str(v)[:500] if not isinstance(v, (int, float, bool)) else v)
                              for k, v in data.items()}}
    node_records.append(rec)
nodes_df = pd.DataFrame(node_records)

edge_records = []
for u, v, key, data in G.edges(keys=True, data=True):
    rec = {"source": u, "target": v, "edge_key": key,
           **{k: (str(v2)[:500] if not isinstance(v2, (int, float, bool)) else v2)
              for k, v2 in data.items()}}
    edge_records.append(rec)
edges_df = pd.DataFrame(edge_records)

conflicts_df = pd.DataFrame(conflicts)

# Recompute node_type_counts on the FINAL graph state
type_counts = pd.Series(
    [d.get("node_type") for _, d in G.nodes(data=True)]
).value_counts()

# Save GraphML (NetworkX handles basic types; stringify complex attrs)
G_serial = G.copy()
for n, d in G_serial.nodes(data=True):
    for k, v in list(d.items()):
        if not isinstance(v, (str, int, float, bool)) and v is not None:
            d[k] = str(v)
for u, v, k, d in G_serial.edges(keys=True, data=True):
    for kk, vv in list(d.items()):
        if not isinstance(vv, (str, int, float, bool)) and vv is not None:
            d[kk] = str(vv)

graphml_path = NB_OUTPUT / "evidence_graph.graphml"
nx.write_graphml(G_serial, graphml_path)
print(f"Saved: {graphml_path}")

nodes_df.to_csv(NB_OUTPUT / "evidence_nodes.csv", index=False)
edges_df.to_csv(NB_OUTPUT / "evidence_edges.csv", index=False)
conflicts_df.to_csv(NB_OUTPUT / "conflicts.csv", index=False)
print(f"Saved: evidence_nodes.csv ({len(nodes_df)} rows)")
print(f"Saved: evidence_edges.csv ({len(edges_df)} rows)")
print(f"Saved: conflicts.csv ({len(conflicts_df)} rows)")
print(f"\nFinal node type counts (recomputed after all mutations):")
print(type_counts.to_string())

In [ ]:
# ============================================================
# QC report + visualization (no downcasting warnings)
# ============================================================
# Force boolean dtype to avoid pandas FutureWarning about downcasting
inj_mask = conflicts_df.get("is_injected_test", pd.Series([False] * len(conflicts_df)))
inj_mask = inj_mask.fillna(False).astype(bool)

qc = {
    "notebook": NOTEBOOK,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "n_patients": int(len(patient_set)),
    "n_cases": int(len(manifest)),
    "graph_nodes": int(G.number_of_nodes()),
    "graph_edges": int(G.number_of_edges()),
    "node_type_counts": type_counts.to_dict() if len(type_counts) else {},
    "n_conflicts_total": int(len(conflicts_df)),
    "n_conflicts_real": int((~inj_mask).sum()) if len(conflicts_df) else 0,
    "n_conflicts_injected": int(inj_mask.sum()) if len(conflicts_df) else 0,
    "conflict_type_counts": (
        conflicts_df["conflict_type"].value_counts().to_dict()
        if len(conflicts_df) else {}
    ),
    "injected_tests_passed": int(test_results_df["passed"].sum()) if len(test_results_df) else 0,
    "injected_tests_total": int(len(test_results_df)),
    "caps": {
        "mutation_node_cap_per_patient": MAX_MUT_NODES_PER_PATIENT,
        "resistance_node_cap_per_patient": RESISTANCE_NODE_CAP_PER_PATIENT,
    },
    "limitations_carried_forward": [
        "CIViC matches are predominantly gene-level (NB02 civic_data_limitation.json)",
        "Clinical treatment history limited to neoadjuvant flag (NB03 clinical_data_limitation.json)",
        "No guideline or recommendation nodes yet (created in NB06/07)",
        "Conflict rules are heuristic consistency checks, not clinical truth",
        "Mutation and resistance nodes are capped; full tables live in NB02 CSVs",
        "Injected conflict tests are tagged via is_injected_test=True in conflicts.csv",
    ],
    "patient_leakage_note": f"All nodes/edges keyed by patient_id; {N_PATIENTS} patients x {N_CASES} cases",
}

with open(NB_OUTPUT / "evidence_graph_qc_report.json", "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2, default=str)

print(json.dumps({k: v for k, v in qc.items() if not isinstance(v, dict)}, indent=2))
print("\nFinal node type counts:")
print(type_counts.to_string())

# Visualization: conflict counts by type
if len(conflicts_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    real = conflicts_df[~inj_mask]
    if len(real):
        real["conflict_type"].value_counts().plot(kind="barh", ax=axes[0], color="steelblue")
        axes[0].set_title("Real conflicts by type")
        axes[0].set_xlabel("Count")
    sev = real["severity"].value_counts() if len(real) else pd.Series(dtype=int)
    if len(sev):
        sev.plot(kind="bar", ax=axes[1], color="coral")
        axes[1].set_title("Real conflicts by severity")
        axes[1].set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(NB_OUTPUT / "conflict_summary.png", dpi=150, bbox_inches="tight")
    plt.show()

# Node-type bar (uses FINAL type_counts)
fig, ax = plt.subplots(figsize=(8, 5))
type_counts.plot(kind="barh", ax=ax, color="seagreen")
ax.set_title("Evidence graph node types (final)")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig(NB_OUTPUT / "graph_node_types.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# DOCUMENTATION — graph & conflict limitations
# ============================================================
limitations = {
    "graph_scope": (
        "Graph covers patient, case, WSI, pathology summary, mutation (capped), gene, "
        "gene_summary, CIViC evidence, resistance signal, clinical finding/summary, "
        "and conflict nodes. Recommendation / guideline / trial nodes are deferred "
        "to Notebooks 06–07."
    ),
    "mutation_node_cap": (
        f"At most {MAX_MUT_NODES_PER_PATIENT} mutation nodes per patient are materialised "
        "to keep the graph tractable on Kaggle. Full mutation tables remain in NB02 CSVs."
    ),
    "resistance_node_cap": (
        f"At most {RESISTANCE_NODE_CAP_PER_PATIENT} resistance-signal nodes per patient "
        "are materialised. The true match count is preserved in the "
        "`resistance_signal_present` conflict as `true_resistance_count`. "
        "Full list is in NB02 resistance_evidence.csv."
    ),
    "injected_conflict_semantics": (
        "Injected test conflicts are tagged with is_injected_test=True and are "
        "also materialised as graph nodes with the same flag for auditability. "
        "They exist for unit-testing the detector and are not real patient evidence."
    ),
    "conflict_semantics": (
        "Conflicts are consistency / completeness flags, not adjudicated clinical "
        "disagreements. They must not be presented as proven medical contradictions."
    ),
    "civic_limitation": civic_lim.get("civic_granularity", "see NB02") if civic_lim else "see NB02",
    "clinical_limitation": (
        "This notebook consumed the NB03 dataset version available at run time. "
        "If NB03 was re-run with extended diagnosis/biomarker fields, richer "
        "diagnosis and biomarker nodes will be present. "
        "See NB03 clinical_data_limitation.json."
    ),
    "no_auto_resolution": (
        "Conflicts are recorded only; resolution is left to human MDT / later agents."
    ),
    "downstream_requirements": [
        "NB05 must treat resistance_signal and civic_gene_only flags as prompts "
        "for counter-evidence retrieval.",
        "NB06 planners must read conflicts.csv and must not ignore high-severity "
        "missing_stage / missing demographics.",
        "NB07 red-team should prioritise patients with resistance_signal_present "
        "or high_mutation_burden.",
        "NB08 safety gate should escalate on high-severity conflicts when combined "
        "with high uncertainty.",
    ],
}
with open(NB_OUTPUT / "evidence_graph_limitation.json", "w", encoding="utf-8") as f:
    json.dump(limitations, f, indent=2)

print("=" * 60)
print("EVIDENCE GRAPH LIMITATION SUMMARY")
print("=" * 60)
for k, v in limitations.items():
    if isinstance(v, list):
        print(f"  {k}: {len(v)} items")
    else:
        print(f"  {k}: {str(v)[:90]}...")
print("=" * 60)

In [ ]:
# ============================================================
# OUTPUTS — final inventory & completion banner
# ============================================================
import shutil

outputs = []
for p in sorted(NB_OUTPUT.rglob("*")):
    if p.is_file():
        outputs.append({
            "file":       str(p.relative_to(NB_OUTPUT)),
            "size_bytes": p.stat().st_size,
        })
display(pd.DataFrame(outputs))

print("\n" + "=" * 50)
print("VERIONCO NOTEBOOK 04 COMPLETE")
print("=" * 50)
print(f"Artifacts: {NB_OUTPUT}")

shutil.make_archive(str(OUTPUT_ROOT / "04_artifacts"), "zip", str(NB_OUTPUT))
print(f"Zipped:    {OUTPUT_ROOT / '04_artifacts.zip'}")

## Handoff to Notebook 05 (Retrieval — Support & Counter)

**Inputs from Notebook 04:**
- `evidence_graph.graphml` — full provenance-aware graph
- `evidence_nodes.csv` / `evidence_edges.csv` — tabular views
- `conflicts.csv` — real + injected conflicts with severity and type
- `injected_conflict_tests.json` — unit-test cases for the detector
- `evidence_graph_qc_report.json` — includes caps and final node-type counts
- `evidence_graph_limitation.json` — documented graph/data limitations

**Leakage contract:** every node and conflict carries `patient_id`.
**Critical flags for NB05:** `resistance_signal_present`, `civic_gene_only`, `high_mutation_burden`.

Notebook 05 must implement **separate** support-retrieval and counter-retrieval
channels; it must not simply reuse the same hits for both objectives.

**Caps to respect:** `caps.mutation_node_cap_per_patient` and
`caps.resistance_node_cap_per_patient` from the QC report — downstream must
read the full lists from NB02 CSVs if the graph is insufficient.

In [ ]:
# ============================================================
# EXPORT & DOWNLOAD ARTIFACT BUNDLE
# ============================================================
import shutil
from pathlib import Path
try:
    from IPython.display import FileLink, display
except ImportError:
    def FileLink(p): return p
    def display(*a): pass

export_dir = OUTPUT_ROOT / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
zip_name = f'verionco_outputs_{NB_OUTPUT.name}'
zip_path = export_dir / zip_name

# Package NB_OUTPUT into zip
shutil.make_archive(str(zip_path), 'zip', NB_OUTPUT)
final_zip = export_dir / f'{zip_name}.zip'

print('=' * 60)
print(f'VERIONCO NOTEBOOK {NB_OUTPUT.name} COMPLETE')
print(f'Artifact bundle created: {final_zip}')
print(f'Archive size: {final_zip.stat().st_size / (1024*1024):.2f} MB')
print('=' * 60)

# Display download link (clickable in Kaggle/Jupyter)
display(FileLink(str(final_zip)))
